# 从 `Branch` 到 `Cell/CV`：当前用户入口教程

这份 notebook 只讲 **当前仓库已经可用** 的入口能力，主线如下：

1. `braincell.morpho.Branch`：分支几何原语
2. `braincell.morpho.Morpho / MorphoBranch`：形态树组织与查询
3. `braincell.cell.Cell / CV / CVPolicy`：前端声明与离散视图
4. 与 `HHTypedNeuron` 可执行运行时之间的差距

> 说明：当前 `Cell` 是 frontend-only（声明与离散视图），不是可直接 `run()` 的神经元。


## 0. 环境与导入

如果你的机器没有可用 GPU，先固定 `JAX_PLATFORMS=cpu`，避免导入 `braincell` 时触发 CUDA backend 错误。


In [1]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")

import braincell
import brainunit as u

from braincell import (
    Branch,
    Morpho,
    Cell,
    CVPolicy,
    CableProperties,
    CurrentClamp,
    DensityMechanism,
)
from braincell.filter import BranchSlice, RootLocation

print("braincell version:", braincell.__version__)


/home/swl/anaconda3/envs/braincell/lib/python3.10/site-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.5.1 is installed, but it is not compatible with the installed jaxlib version 0.6.2, so it will not be used.
  warnings.warn(


braincell version: 0.0.8


## 1. `Branch`：几何层最小原语

`Branch` 只负责一条分支的几何，不负责拓扑连接。

当前常用构造方式：

- `Branch.lengths_shared(...)`
- `Branch.lengths_paired(...)`
- `Branch.xyz_shared(...)`
- `Branch.xyz_paired(...)`


In [2]:
# 1) 段长 + 连续半径
soma = Branch.lengths_shared(
    lengths=[20.0],
    radii=[10.0, 10.0],
    type="soma",
)

# 2) 段长 + 每段近/远端半径对
basal = Branch.lengths_paired(
    lengths=[40.0, 30.0],
    radii_pairs=[[2.0, 1.8], [1.8, 1.2]],
    type="basal_dendrite",
)

# 3) 三维点 + 连续半径
apical = Branch.xyz_shared(
    points=[[0.0, 0.0, 0.0], [15.0, 0.0, 0.0], [30.0, 5.0, 0.0]] * u.um,
    radii=[1.5, 1.2, 0.9] * u.um,
    type="apical_dendrite",
)

print("soma segments:", soma.n_segments)
print("basal segments:", basal.n_segments)
print("apical segments:", apical.n_segments)


soma segments: 1
basal segments: 2
apical segments: 2


In [3]:
print("soma total_length (um):", float(soma.total_length.to_decimal(u.um)))
print("basal lateral areas n:", len(basal.lateral_areas()))
print("basal volumes n:", len(basal.volumes()))
print("apical distal radius (um):", float(apical.radius_distal.to_decimal(u.um)))


soma total_length (um): 20.0
basal lateral areas n: 2
basal volumes n: 2
apical distal radius (um): 0.8999999761581421


## 2. `Morpho / MorphoBranch`：树结构与拓扑查询

`Morpho` 管理分支连接关系；`MorphoBranch` 是树内某个分支的视图对象。

这里演示：

- `Morpho.from_root(...)` 创建根
- `tree.soma.xxx = branch` 语法糖挂载
- `tree.attach(...)` 显式挂载
- `topo()` / `branch_by_name()` / `path_to_root()` 等查询


In [4]:
tree = Morpho.from_root(soma, name="soma")

tree.soma.basal_slot = basal
# 显式指定 parent_x 挂一个 apical 分支
tree.attach(parent="soma", child="apical_slot", branch=apical, parent_x=0.7)

print(tree.topo())
print("n_branches:", len(tree.branches))
print("n_connections:", len(tree.connections))

apical_view = tree.branch_by_name("apical_dendrite_0")
print("apical index:", apical_view.index)
print("apical path_to_root:", tree.path_to_root(apical_view.index))


soma
├── basal_dendrite_0
└── apical_dendrite_0
n_branches: 3
n_connections: 2
apical index: 2
apical path_to_root: (0, 2)


In [5]:
root = tree.soma
print("root name:", root.name)
print("root parent:", root.parent)
print("root children:", [c.name for c in root.children])

first_child = root.children[0]
print("first child name:", first_child.name)
print("first child parent_x:", first_child.parent_x)
print("first child child_x:", first_child.child_x)


root name: soma
root parent: None
root children: ['basal_dendrite_0', 'apical_dendrite_0']
first child name: basal_dendrite_0
first child parent_x: 1.0
first child child_x: 0.0


## 3. `Cell / CV / CVPolicy`：frontend 声明与离散结果

`Cell` 会在内部复制一份 `Morpho` 快照，并根据 `CVPolicy` 生成 `CV` 视图。

你现在能稳定拿到：

- `cell.summary()`
- `cell.n_cv`
- `cell.cv(i)`（几何、电缆属性、机制容器等）


In [20]:
cell = Cell(tree)
print("cell summary:", cell.summary())
print("cell n_cv:", cell.n_cv)

cv0 = cell.cv(0)
print("cv0.id:", cv0.id)
print("cv0.region:", cv0.region)
print("cv0.length (um):", cv0.length)
print("cv0.lateral_area (um^2):", cv0.lateral_area)
print("cv0.v (mV):", cv0.v)
print("cv0.temp (K):", cv0.temp)
cell.cvs


cell summary: {'n_cv': 3, 'n_paint_rules': 1, 'n_place_rules': 0}
cell n_cv: 3
cv0.id: 0
cv0.region: RegionMask(intervals=((0, 0.0, 1.0),))
cv0.length (um): 20. * umetre
cv0.lateral_area (um^2): 1256.6371 * umeter2
cv0.v (mV): -65. * mvolt
cv0.temp (K): 309.15 * kelvin


(CV(id=0, branch_id=0, branch_type='soma', prox=0.0, dist=1.0, parent_cv=None, children_cv=(1, 2), length=20. * umetre, lateral_area=1256.6371 * umeter2, cm=1. * ufarad / cmeter2, ra=100. * ohm * cmetre, v=-65. * mvolt, temp=309.15 * kelvin, r_axial=63661.977 * ohm, r_axial_prox=31830.988 * ohm, r_axial_dist=31830.988 * ohm, density_mech=(), point_mech=(), _frusta=(CVFrustum(prox=0.0, dist=1.0, length=20. * umetre, radius_prox=10. * umetre, radius_dist=10. * umetre, point_prox=None, point_dist=None),)),
 CV(id=1, branch_id=1, branch_type='basal_dendrite', prox=0.0, dist=1.0, parent_cv=0, children_cv=(), length=70. * umetre, lateral_area=760.32794 * umeter2, cm=1. * ufarad / cmeter2, ra=100. * ohm * cmetre, v=-65. * mvolt, temp=309.15 * kelvin, r_axial=7957747. * ohm, r_axial_prox=3052286.5 * ohm, r_axial_dist=4905460.5 * ohm, density_mech=(), point_mech=(), _frusta=(CVFrustum(prox=0.0, dist=0.5714285714285714, length=40. * umetre, radius_prox=2. * umetre, radius_dist=1.8 * umetre, poin

In [21]:
cell_dense = Cell(tree, cv_policy=CVPolicy(cv_per_branch=5))
print("cv_per_branch=2 -> n_cv:", cell_dense.n_cv)

cell_len = Cell(tree, cv_policy=CVPolicy(mode="max_cv_len", max_cv_len=10.0 * u.um))
print("max_cv_len=25um -> n_cv:", cell_len.n_cv)


cv_per_branch=2 -> n_cv: 15
max_cv_len=25um -> n_cv: 13


## 4. `paint / place`：声明式配置

当前 `Cell` 支持积累声明规则：

- `paint(region, mechanisms...)`
- `place(locset, point_mechanisms...)`

注意：这一步仍是 frontend 声明与离散映射，不是运行仿真。


In [8]:
cell_cfg = Cell(tree, cv_policy=CVPolicy(cv_per_branch=2))

# 在 soma 中间区域覆盖 cable 属性
cell_cfg.paint(
    BranchSlice(branch_index=0, prox=0.4, dist=0.6),
    CableProperties(
        resting_potential=-70.0 * u.mV,
        membrane_capacitance=1.5 * (u.uF / u.cm**2),
        axial_resistivity=120.0 * (u.ohm * u.cm),
        temperature=u.celsius2kelvin(30.0),
    ),
)

# 在 basal 分支区域添加一个密度机制声明
cell_cfg.paint(
    BranchSlice(branch_index=1, prox=0.0, dist=1.0),
    DensityMechanism(channel_type="leaky", params=(("g_max", 0.2 * (u.mS / u.cm**2)),)),
)

# 在 root 位点放置点机制
cell_cfg.place(
    RootLocation(x=0.5),
    CurrentClamp(amplitude=0.05 * u.nA, delay=1.0 * u.ms, duration=2.0 * u.ms),
)

print("after paint/place summary:", cell_cfg.summary())
for i in range(cell_cfg.n_cv):
    cv = cell_cfg.cv(i)
    print(
        f"CV {i}: branch={cv.branch_id}, point_mech={len(cv.point_mech)}, "
        f"density_mech={len(cv.density_mech)}, v={float(cv.v.to_decimal(u.mV)):.2f}mV"
    )


after paint/place summary: {'n_cv': 6, 'n_paint_rules': 3, 'n_place_rules': 1}
CV 0: branch=0, point_mech=0, density_mech=0, v=-65.00mV
CV 1: branch=0, point_mech=1, density_mech=0, v=-65.00mV
CV 2: branch=1, point_mech=0, density_mech=1, v=-65.00mV
CV 3: branch=1, point_mech=0, density_mech=1, v=-65.00mV
CV 4: branch=2, point_mech=0, density_mech=0, v=-65.00mV
CV 5: branch=2, point_mech=0, density_mech=0, v=-65.00mV


In [16]:
cell_cfg.cv(2).density_mech

(DensityMechanism(ion_type=None, channel_type='leaky', params=(('g_max', 0.2 * msiemens / cmeter2),)),)

In [9]:
# CV 也可以回退成一个 Branch 视图（便于几何层复用）
slice_branch = cell_cfg.cv(1).as_branch()
print("slice_branch type:", slice_branch.type)
print("slice_branch n_segments:", slice_branch.n_segments)
print("slice_branch total_length (um):", float(slice_branch.total_length.to_decimal(u.um)))


slice_branch type: soma
slice_branch n_segments: 1
slice_branch total_length (um): 10.0


## 5. 目前到 `HHTypedNeuron` 的差距

下面是你现在最关心的边界：

### 已有（可以直接用）

- `Branch/Morpho`：形态几何与拓扑组织
- `Cell/CV`：离散视图、`paint/place` 声明、规则查询
- 机制数据容器：`CableProperties / DensityMechanism / CurrentClamp` 等

### 仍缺（尚未贯通）

- `Cell` -> 可执行 runtime 的 compile/lower 管线
- 与 `_base.py` 中 `HHTypedNeuron` 的直接桥接对象
- 基于 `quad` 的端到端 `run()` 仿真入口（多房室路径）
- 轴向矩阵求解与通道/离子状态在同一执行图中的统一调度

### 结论

当前 `Cell` 是“声明 + 离散化 + 可检查视图”层，不是“可运行多房室仿真”层。
如果你现在要跑仿真，仍应使用已成熟的 `SingleCompartment` 路径。


## 6. 下一步建议

- 如果目标是文档：继续补 `filter` 选择器与 `paint/place` 的映射示例。
- 如果目标是执行：先定义 `Cell -> HHTypedNeuron` 的最小编译产物（状态、矩阵、通道实例化），再接 `quad`。
